# Southern Tree Species
Species
American Basswood
American Beech
American Elm
American sycamore 
Bald Cypress
Bigleaf Magnolia
Bitternut Hickory
Black Cherry
Black locust 
Black Oak 
Black walnut 
Black Willow
Blackgum 
Boxelder
Cherrybark Oak
Chestnut Oak
Chinkapin oak 
Common persimmon 
Eastern cottonwood 
Eastern Hophornbeam
Eastern Redbud
Eastern Redcedar
Eastern White Pine
Florida Maple
Flowering Dogwood
Green Ash
Hackberry
Honeylocust
Live Oak
Loblolly Pine
Longleaf Pine
Mimosa
Mockernut Hickory
Musclewood
Northern Red Oak
Ohio Buckeye
Osage-orange
Overcup Oak
Pecan
Pignut Hickory
Pin Oak 
Post Oak
Red Maple
River Birch
Sassafras
Sawtooth Oak
Scarlet Oak
Serviceberry
Shagbark hickory
shingle oak
Shortleaf Pine
silver maple
Slash Pine
slippery elm
Sourwood
Southern Magnolia
Southern Red Oak
sugar maple
Sugarberry
Swamp Chestnut Oak
Sweetbay Magnolia
sweetgum
Virginia Pine
Water Oak
White Ash
White Oak
Willow Oak
Winged Elm
Yellow Poplar

The following code block imports the labels CSV as a Pandas dataframe. It then displays the first five rows of the dataframe and prints the original row count.

The code filters out rows where the species was listed as "other" and prints the remaining row count.

Afterward, the Species column is selected as the target label, while the remaining columns are stored separately as input features. These inputs and labels are then split into training and testing sets. For initial exploration, only 5% of the data is used for training so that model runs stay fast

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

import os

IN_COLAB = os.path.exists("/content")

if IN_COLAB:
    import importlib

    if not os.path.exists("/content/drive/MyDrive"):
        drive = importlib.import_module("google.colab.drive")
        drive.mount("/content/drive")
    
    BASE_PATH = "/content/drive/MyDrive/Bark_Imagery/"
else:
    BASE_PATH = ""

labels_path = BASE_PATH + "bark_images_and_labels/bark_class_labels.csv"
extract_path = BASE_PATH + "bark_images_and_labels/BARK_UGA_PICS/"

df = pd.read_csv(labels_path)
display(df.head())
print("Original dataframe:", df.shape[0])

df = df[~df['Species'].str.lower().eq('other')].copy()
print("Dataframe after removing 'other' species:", df.shape[0])

print("Number of unique species:", df['Species'].nunique())

sample_tree_df, remaining_tree_df = train_test_split(
    df, 
    train_size=0.05, 
    random_state=42,
    stratify=df["Species"]
)
print("Training set size:", sample_tree_df.shape[0])
print("Number of unique species in training set:", sample_tree_df['Species'].nunique())

print("Remaining set size:", remaining_tree_df.shape[0])
print("Number of unique species in remaining set:", remaining_tree_df['Species'].nunique())

Mounted at /content/drive


,OBJECTID,Species,Moisture_Condition,DBH_in,Note,CreatedBy,CreateTime,EditedBy,EditTime,Photo0,Photo1,Photo2,Photo3
0,1,Pin oak,Dry,17.0,NaN,tdb96463_USG,2024/02/05 18:42:18.856+00,tdb96463_USG,2024/02/05 18:42:18.856+00,ATT1_Photo 3.jpg,ATT2_Photo 2.jpg,ATT3_Photo 4.jpg,ATT4_Photo 1.jpg
1,2,Eastern white pine,Dry,29.0,NaN,tdb96463_USG,2024/02/05 18:49:21.604+00,tdb96463_USG,2024/02/05 18:49:21.604+00,ATT5_Photo 3.jpg,ATT6_Photo 2.jpg,ATT7_Photo 4.jpg,ATT8_Photo 1.jpg
2,3,Yellow-poplar,Dry,18.0,NaN,tdb96463_USG,2024/02/05 18:50:52.876+00,tdb96463_USG,2024/02/05 18:50:52.876+00,ATT9_Photo 2.jpg,ATT10_Photo 3.jpg,ATT11_Photo 1.jpg,ATT12_Photo 4.jpg
3,4,Loblolly pine,Dry,16.0,NaN,tdb96463_USG,2024/03/08 20:53:15.938+00,tdb96463_USG,2024/03/08 20:53:15.938+00,ATT13_Photo 2.jpg,ATT14_Photo 3.jpg,ATT15_Photo 4.jpg,ATT16_Photo 1.jpg
4,5,Loblolly pine,Dry,17.0,NaN,tdb96463_USG,2024/03/08 20:56:20.335+00,tdb96463_USG,2024/03/08 20:56:20.335+00,ATT17_Photo 1.jpg,ATT18_Photo 4.jpg,ATT19_Photo 3.jpg,ATT20_Photo 2.jpg


Original dataframe: 692
Dataframe after removing 'other' species: 685
Number of unique species: 25
Training set size: 34
Number of unique species in training set: 21
Remaining set size: 651
Number of unique species in remaining set: 25


In [3]:
species_counts_df = (
    df["Species"]
    .value_counts()
    .rename_axis("Species")
    .reset_index(name="TreeCount")
)

display(species_counts_df)

print("Number of species:", species_counts_df.shape[0])
print("Total trees:", species_counts_df["TreeCount"].sum())

,Species,TreeCount
0,Loblolly pine,172
1,Southern red oak,54
2,Willow oak,53
3,Live oak,50
4,Water oak,35
5,Scarlet oak,31
6,Swamp chestnut oak,29
7,Mockernut hickory,27
8,Bald cypress,22
9,Pecan,22


Number of species: 25
Total trees: 685


The following code block melts the sampled tree-level dataframe across the four photo columns. This converts the data from one row per tree, with separate Photo0, Photo1, Photo2, and Photo3 columns, into one row per image with a single Image column.

The OBJECTID and Species columns are preserved so that each melted image row still keeps its tree identifier and species label. Rows with missing image filenames are removed, and the index is reset.

The code then builds a full ImagePath column by combining the extracted image folder path with each filename in the Image column. It verifies that all images for tree 598 remain together in the sampled dataframe, displays the sample image count and first few rows, and checks whether the generated image paths exist on disk.

In [4]:
import os

photo_labels = ["Photo0", "Photo1", "Photo2", "Photo3"]

def melt_photos(split_df):
    melted_df = split_df.melt(
        id_vars=['OBJECTID', 'Species'], 
        value_vars=photo_labels, 
        var_name='Photo', 
        value_name='Image'
    )
    
    return melted_df.dropna(subset=['Image']).reset_index(drop=True)

def build_image_paths(image_df, extract_path):
    image_df['ImagePath'] = extract_path + image_df['Image']
    return image_df

sample_image_df = melt_photos(sample_tree_df)
sample_image_df = build_image_paths(sample_image_df, extract_path)
path_exists_df = sample_image_df["ImagePath"].apply(os.path.exists)

print("Verification of tree level split using tree number 598")
display(sample_image_df[sample_image_df["OBJECTID"] == 598])

print("Sample set image count:", sample_image_df.shape[0])
display(sample_image_df.head())

display(path_exists_df.head())
print(path_exists_df.value_counts())

Verification of tree level split using tree number 598


,OBJECTID,Species,Photo,Image,ImagePath
3,598,Water oak,Photo0,ATT2385_Photo 2.jpg,/content/drive/MyDrive/Bark_Imagery/bark_image...
37,598,Water oak,Photo1,ATT2386_Photo 4.jpg,/content/drive/MyDrive/Bark_Imagery/bark_image...
71,598,Water oak,Photo2,ATT2387_Photo 1.jpg,/content/drive/MyDrive/Bark_Imagery/bark_image...
105,598,Water oak,Photo3,ATT2388_Photo 3.jpg,/content/drive/MyDrive/Bark_Imagery/bark_image...


Sample set image count: 136


,OBJECTID,Species,Photo,Image,ImagePath
0,40,Southern magnolia,Photo0,ATT157_Photo 4.jpg,/content/drive/MyDrive/Bark_Imagery/bark_image...
1,54,Longleaf pine,Photo0,ATT213_Photo 2.jpg,/content/drive/MyDrive/Bark_Imagery/bark_image...
2,615,Eastern white pine,Photo0,ATT2453_Photo 4.jpg,/content/drive/MyDrive/Bark_Imagery/bark_image...
3,598,Water oak,Photo0,ATT2385_Photo 2.jpg,/content/drive/MyDrive/Bark_Imagery/bark_image...
4,203,Willow oak,Photo0,ATT809_Photo 3.jpg,/content/drive/MyDrive/Bark_Imagery/bark_image...


,ImagePath
0,True
1,True
2,True
3,True
4,True


ImagePath
True    136
Name: count, dtype: int64


The following code block defines the folder path where the extracted bark images are stored. It then defines a function that creates a new ImagePath column by combining the base image folder path with each image filename from the Image column.

The function is applied to both the training and testing image dataframes. Finally, the first few rows of the training dataframe are displayed to verify that the image paths were created correctly.

In [5]:
import torch
from torchvision import transforms
from sklearn.preprocessing import LabelEncoder
from PIL import Image
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import torch.nn as nn
import gc
import numpy as np
import os

class BarkDataset(Dataset):
        def __init__(self, image_df, transform=None):
            self.image_df = image_df
            self.transform = transform
        
        def __len__(self):
            return len(self.image_df)
        
        def __getitem__(self, index):
            row = self.image_df.iloc[index]
            image_path = row["ImagePath"]
            label = row["Label"]

            try:
                image = Image.open(image_path).convert("RGB")
            except Exception as e:
                raise RuntimeError(f"Error loading image at {image_path}") from e

            
            if self.transform is not None:
                image = self.transform(image)

            label = torch.tensor(label, dtype=torch.long)
            
            return image, label

class TrainTestBarkModel():
    """
        class is initialized with default values, image_df or tree_df must be provided, but not both. 
        If image_df is provided, it will be used directly for training. 
        If tree_df is provided, it will be split into train, val, and test sets, and then melted to create the image_df for each set. 
        The label encoding is done on the entire dataset to ensure consistent mapping across splits. 
        The melt_labels and extract_path can be customized if needed.
    """
    def __init__(
            self, 
            train_image_df=None,
            val_image_df=None,
            test_image_df=None,
            tree_df=None,
            melt_labels=["Photo0", "Photo1", "Photo2", "Photo3"],
            extract_path = 'bark_images_and_labels/BARK_UGA_PICS/',
            data_fraction=1.0,
            train_size=0.7,
            val_size=0.15,
            test_size=0.15,
            random_state=42
        ):

        self.train_image_df = train_image_df
        self.val_image_df = val_image_df
        self.test_image_df = test_image_df

        if (train_image_df is None and tree_df is None) or (
            tree_df is not None and (
                train_image_df is not None 
                or val_image_df is not None 
                or test_image_df is not None
            )
        ):
            raise ValueError(
                "Either train_image_df or tree_df must be provided, but not both.\n"
                "val_image_df and test_image_df are optional but should only be provided with train_image_df."
            )
        elif train_image_df is not None:
            self.train_image_df = train_image_df.reset_index(drop=True).copy()
            label_encoder = LabelEncoder()
            self.train_image_df['Label'] = label_encoder.fit_transform(self.train_image_df['Species'])
            self.set_data_info(self.train_image_df)
        elif tree_df is not None:
            self.tree_df = tree_df.reset_index(drop=True).copy()
            self.melt_labels = melt_labels.copy()
            self.extract_path = extract_path
            self.data_fraction = data_fraction
            self.train_size = train_size
            self.val_size = val_size
            self.test_size = test_size
            self.random_state = random_state
            self.train_image_df, self.val_image_df, self.test_image_df = self.split_data(self.tree_df)
            self.set_data_info(self.train_image_df)

    """
        Sets the number of classes.
        Creates a label map dataframe based on the provided image_df.
    """
    def set_data_info(self, image_df):
        self.num_classes = image_df['Label'].nunique()

        self.label_map_df = (
            image_df[["Label", "Species"]]
            .drop_duplicates()
            .sort_values("Label")
            .reset_index(drop=True)
        )

    """
        This def is called by the class initializer if a tree_df is provided. It performs the following steps:
        1. Validates the data_fraction and split sizes.
        2. If data_fraction is less than 1.0, it randomly samples the specified fraction of the tree_df while maintaining the class distribution.
        3. Splits the selected tree_df into train and temp sets based on the specified train_size, maintaining class distribution.
        4. Calculates the relative size for the test set and splits the temp set into val and test sets, maintaining class distribution.
        5. Fits a label encoder on the entire tree_df to ensure consistent mapping across splits and applies the encoding to the train, val, and test sets.
        6. Melts the photo columns for each split to create the image_df for train, val, and test sets.
        7. Builds the image paths for each split by concatenating the extract_path with the image filenames.
        8. Returns the train_image_df, val_image_df, and test_image_df.
    """
    def split_data(self, tree_df):
        if not (0.0 < self.data_fraction <= 1.0):
            raise ValueError("Data fraction must be between 0.0 and 1.0")
        
        if not np.isclose(self.train_size + self.val_size + self.test_size, 1.0):
            raise ValueError("Train, validation, and test sizes must sum to 1.0")
        
        # Based on the requested data_fraction, train_size, val_size, and test_size, does each species probably have enough trees before trying to split?
        temp_size = self.val_size + self.test_size

        min_trees_needed = int(np.ceil(2 / (self.data_fraction * temp_size)))

        species_counts = tree_df["Species"].value_counts()
        too_small_species = species_counts[species_counts < min_trees_needed]

        if len(too_small_species) > 0:
            raise ValueError(
                "Some species do not have enough trees for this stratified split.\n"
                f"With data_fraction={self.data_fraction}, train_size={self.train_size}, "
                f"val_size={self.val_size}, and test_size={self.test_size}, each species needs "
                f"at least {min_trees_needed} trees.\n\n"
                f"Too-small species:\n{too_small_species}"
            ) 

        if self.data_fraction < 1.0:
            selected_tree_df, _ = train_test_split(
                tree_df,
                train_size=self.data_fraction,
                random_state=self.random_state,
                stratify=tree_df["Species"]
            )
        else:
            selected_tree_df = tree_df.copy()

        train_tree_df, temp_tree_df = train_test_split(
            selected_tree_df,
            train_size=self.train_size,
            random_state=self.random_state,
            stratify=selected_tree_df["Species"]
        )

        # After the initial train split, check if the temp_tree_df has at least 2 trees per species before trying to split into val and test
        temp_species_counts = temp_tree_df["Species"].value_counts()
        too_small_temp_species = temp_species_counts[temp_species_counts < 2]

        if len(too_small_temp_species) > 0:
            raise ValueError(
                "Some species have fewer than 2 trees in the validation/test pool after splitting.\n"
                "Increase data_fraction, lower train_size, or remove rare species.\n\n"
                f"Too-small temp species:\n{too_small_temp_species}"
            )

        # Calculate the relative size of the test set with respect to the temp set to ensure the correct proportions for val and test splits
        test_relative_size = self.test_size / (self.val_size + self.test_size)

        val_tree_df, test_tree_df = train_test_split(
            temp_tree_df,
            test_size=test_relative_size,
            random_state=self.random_state,
            stratify=temp_tree_df["Species"]
        )

        #fit label encoder on the entire dataset to ensure consistent mapping across splits
        label_encoder = LabelEncoder()
        label_encoder.fit(tree_df['Species'])

        #apply the same label encoding to all splits
        train_tree_df['Label'] = label_encoder.transform(train_tree_df['Species'])
        val_tree_df['Label'] = label_encoder.transform(val_tree_df['Species'])
        test_tree_df['Label'] = label_encoder.transform(test_tree_df['Species'])

        train_image_df = self.melt_photos(train_tree_df)
        val_image_df = self.melt_photos(val_tree_df)
        test_image_df = self.melt_photos(test_tree_df)

        train_image_df = self.build_image_paths(train_image_df)
        val_image_df = self.build_image_paths(val_image_df)
        test_image_df = self. build_image_paths(test_image_df)

        return train_image_df, val_image_df, test_image_df

    """
        Utility function to melt the photo columns into a long format dataframe with one row per image.
        The function takes a split_df (train, val, or test) and melts the specified photo columns into a 
        long format dataframe with columns for OBJECTID, Species, Photo, and Image.
    """
    def melt_photos(self, split_df):
        melted_df = split_df.melt(
            id_vars=['OBJECTID', 'Species', 'Label'], 
            value_vars=self.melt_labels, 
            var_name='Photo', 
            value_name='Image'
        )
        
        return melted_df.dropna(subset=['Image']).reset_index(drop=True)
    
    """
        Utility function to build the full image paths by concatenating the extract_path with the image filenames.
        The function takes an image_df with an 'Image' column containing the image filenames and adds a new 'ImagePath' 
        column with the full path to each image.
    """
    def build_image_paths(self, image_df):
        image_df['ImagePath'] = self.extract_path + image_df['Image']
        return image_df

    """
        Called by user.
        Displays the label to species mapping and the number of classes.
    """
    def show_label_map(self):
        print("Label to Species mapping: ", self.num_classes, " classes")
        display(self.label_map_df)

    def get_num_classes(self):
        return self.num_classes
        
    def get_num_images(self, image_df=None):
        if image_df is None:
            raise ValueError("image_df must be provided to get the number of images.")
        return len(image_df)

    def get_dataloader(self, batch_size, shuffle, image_df=None, transform=None):
        if image_df is None:
            image_df = self.train_image_df

        if transform is None:
            raise ValueError("transform must be provided for training/evaluation dataloaders.")

        dataset = BarkDataset(
            image_df,
            transform=transform
        )

        dataloader = DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=shuffle
        )

        return dataloader
    
    def get_transform(self, weights=None, image_size=None):
        if image_size is None and weights is not None:
            return weights.transforms()

        if image_size is None:
            image_size = (224, 224)

        return transforms.Compose([
            transforms.Resize(image_size),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225],
            )
        ])
    
    def train_model(self, model_tuple, batch_size, num_epochs):
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        if num_epochs == None:
            num_epochs = self.get_num_images(self.train_image_df) // batch_size

        model = model_tuple[1](weights=model_tuple[2])

        train_transform = self.get_transform(weights=model_tuple[2], image_size=model_tuple[3])
        eval_transform = self.get_transform(weights=model_tuple[2], image_size=model_tuple[3])

        if hasattr(model, "fc"):
            model.fc = nn.Linear(
                model.fc.in_features, 
                self.num_classes
            )
        elif hasattr(model, "classifier") and isinstance(model.classifier, nn.Sequential):
            model.classifier[-1] = nn.Linear(
                model.classifier[-1].in_features,
                self.num_classes
            )
        else:
            raise ValueError("Unsupported model architecture")

        model = model.to(device)
        print(f"\n===== Training {model_tuple[0]} =====")
        if self.val_image_df is not None and self.test_image_df is not None:
            print(f"Using datafraction: {self.data_fraction}, train_size: {self.train_size}, val_size: {self.val_size}, test_size: {self.test_size}")
        print(f"Model loaded on device: {device}")
        print(f"Number of classes: {self.num_classes}")

        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

        for epoch in range(num_epochs):
            model.train()

            running_loss = 0.0
            correct = 0
            total = 0

            for images, labels in self.get_dataloader(batch_size=batch_size, shuffle=True, transform=train_transform):
                images, labels = images.to(device), labels.to(device)

                optimizer.zero_grad()

                outputs = model(images)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()

                running_loss += loss.item() * images.size(0)
                predicted_labels = outputs.argmax(dim=1)
                correct += (predicted_labels == labels).sum().item()
                total += labels.size(0)
        
            epoch_loss = running_loss / total
            epoch_accuracy = correct / total

            print(f"Epoch {epoch+1}/{num_epochs}")
            print(f"Loss: {epoch_loss:.4f}")
            print(f"Accuracy: {epoch_accuracy:.4f}")
            print()

        if self.val_image_df is not None:
            val_loss, val_acc = self.evaluate_model(
                model, self.val_image_df, eval_transform, batch_size, "Validation"
            )

        if self.test_image_df is not None:
            test_loss, test_acc = self.evaluate_model(
                model, self.test_image_df, eval_transform, batch_size, "Test"
            )

    def evaluate_model(self, model, image_df, transform, batch_size, split_name):
        device = next(model.parameters()).device
        model.eval()

        criterion = nn.CrossEntropyLoss()

        running_loss = 0.0
        correct = 0
        total = 0

        dataloader = self.get_dataloader(
            batch_size=batch_size,
            shuffle=False,
            image_df=image_df,
            transform=transform
        )

        with torch.no_grad():
            for images, labels in dataloader:
                images, labels = images.to(device), labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)

                running_loss += loss.item() * images.size(0)
                predicted_labels = outputs.argmax(dim=1)
                correct += (predicted_labels == labels).sum().item()
                total += labels.size(0)

        if total == 0:
            raise ValueError(f"{split_name} split has no images.")
        
        avg_loss = running_loss / total
        accuracy = correct / total

        print(f"{split_name} Loss: {avg_loss:.4f}")
        print(f"{split_name} Accuracy: {accuracy:.4f}")
        print()

        return avg_loss, accuracy


    def train_multiple(self, models, batch_size, num_epochs):
        results = []

        for model_tuple in models:
            try:
                self.train_model(
                    model_tuple=model_tuple,
                    batch_size=batch_size,
                    num_epochs=num_epochs,
                )
                results.append({"model": model_tuple[0], "status": "passed", "error": None})
            except (Exception, KeyboardInterrupt) as e:
                print(f"Smoke test failed for {model_tuple[0]}: {e}")
                results.append({"model": model_tuple[0], "status": "failed", "error": str(e)})
            finally:
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

        smoke_results_df = pd.DataFrame(results)
        display(smoke_results_df)

In [6]:
min_trees_per_species = 34

species_counts = df["Species"].value_counts()
kept_species = species_counts[species_counts >= min_trees_per_species].index

df_min_species_twelve = df[df["Species"].isin(kept_species)].copy()

print("Original species:", df["Species"].nunique())
print("Kept species:", df_min_species_twelve["Species"].nunique())
print("Removed species:")
display(species_counts[species_counts < min_trees_per_species])

display(
    df_min_species_twelve["Species"]
    .value_counts()
    .rename_axis("Species")
    .reset_index(name="TreeCount")
)

Original species: 25
Kept species: 5
Removed species:


,count
Species,
Scarlet oak,31
Swamp chestnut oak,29
Mockernut hickory,27
Bald cypress,22
Pecan,22
River birch,21
Sourwood,21
Eastern white pine,19
Shortleaf pine,18


,Species,TreeCount
0,Loblolly pine,172
1,Southern red oak,54
2,Willow oak,53
3,Live oak,50
4,Water oak,35


In [ ]:
from torchvision.models import (
    # ResNet models
    resnet18,
    resnet34,
    resnet50,
    resnet101,
    resnet152,

    # EfficientNet B models
    efficientnet_b0,
    efficientnet_b1,
    efficientnet_b2,
    efficientnet_b3,
    efficientnet_b4,
    efficientnet_b5,
    efficientnet_b6,
    efficientnet_b7,

    # EfficientNet V2 models
    efficientnet_v2_s,
    efficientnet_v2_m,
    efficientnet_v2_l,

    ResNet18_Weights,
    ResNet34_Weights,
    ResNet50_Weights,
    ResNet101_Weights,
    ResNet152_Weights,

    EfficientNet_B0_Weights,
    EfficientNet_B1_Weights,
    EfficientNet_B2_Weights,
    EfficientNet_B3_Weights,
    EfficientNet_B4_Weights,
    EfficientNet_B5_Weights,
    EfficientNet_B6_Weights,
    EfficientNet_B7_Weights,

    EfficientNet_V2_S_Weights,
    EfficientNet_V2_M_Weights,
    EfficientNet_V2_L_Weights   
)

models_default_weights_custom_imagesize_full = [
    # ResNet models
    ("resnet18", resnet18, ResNet18_Weights.DEFAULT, (224, 224)),
    ("resnet34", resnet34, ResNet34_Weights.DEFAULT, (224, 224)),
    ("resnet50", resnet50, ResNet50_Weights.DEFAULT, (224, 224)),
    ("resnet101", resnet101, ResNet101_Weights.DEFAULT, (224, 224)),
    ("resnet152", resnet152, ResNet152_Weights.DEFAULT, (224, 224)),

    # EfficientNet B models
    ("efficientnet_b0", efficientnet_b0, EfficientNet_B0_Weights.DEFAULT, (224, 224)),
    ("efficientnet_b1", efficientnet_b1, EfficientNet_B1_Weights.DEFAULT, (240, 240)),
    ("efficientnet_b2", efficientnet_b2, EfficientNet_B2_Weights.DEFAULT, (288, 288)),
    ("efficientnet_b3", efficientnet_b3, EfficientNet_B3_Weights.DEFAULT, (300, 300)),
    ("efficientnet_b4", efficientnet_b4, EfficientNet_B4_Weights.DEFAULT, (380, 380)),
    ("efficientnet_b5", efficientnet_b5, EfficientNet_B5_Weights.DEFAULT, (456, 456)),
    ("efficientnet_b6", efficientnet_b6, EfficientNet_B6_Weights.DEFAULT, (528, 528)),
    ("efficientnet_b7", efficientnet_b7, EfficientNet_B7_Weights.DEFAULT, (600, 600)),

    # EfficientNet V2 models
    ("efficientnet_v2_s", efficientnet_v2_s, EfficientNet_V2_S_Weights.DEFAULT, (384, 384)),
    ("efficientnet_v2_m", efficientnet_v2_m, EfficientNet_V2_M_Weights.DEFAULT, (480, 480)),
    ("efficientnet_v2_l", efficientnet_v2_l, EfficientNet_V2_L_Weights.DEFAULT, (480, 480)),
]

models_default_weights_full = [
    # ResNet models
    ("resnet18", resnet18, ResNet18_Weights.DEFAULT, None),
    ("resnet34", resnet34, ResNet34_Weights.DEFAULT, None),
    ("resnet50", resnet50, ResNet50_Weights.DEFAULT, None),
    ("resnet101", resnet101, ResNet101_Weights.DEFAULT, None),
    ("resnet152", resnet152, ResNet152_Weights.DEFAULT, None),

    # EfficientNet B models
    ("efficientnet_b0", efficientnet_b0, EfficientNet_B0_Weights.DEFAULT, None),
    ("efficientnet_b1", efficientnet_b1, EfficientNet_B1_Weights.DEFAULT, None),
    ("efficientnet_b2", efficientnet_b2, EfficientNet_B2_Weights.DEFAULT, None),
    ("efficientnet_b3", efficientnet_b3, EfficientNet_B3_Weights.DEFAULT, None),
    ("efficientnet_b4", efficientnet_b4, EfficientNet_B4_Weights.DEFAULT, None),
    ("efficientnet_b5", efficientnet_b5, EfficientNet_B5_Weights.DEFAULT, None),
    ("efficientnet_b6", efficientnet_b6, EfficientNet_B6_Weights.DEFAULT, None),
    ("efficientnet_b7", efficientnet_b7, EfficientNet_B7_Weights.DEFAULT, None),

    # EfficientNet V2 models
    ("efficientnet_v2_s", efficientnet_v2_s, EfficientNet_V2_S_Weights.DEFAULT, None),
    ("efficientnet_v2_m", efficientnet_v2_m, EfficientNet_V2_M_Weights.DEFAULT, None),
    ("efficientnet_v2_l", efficientnet_v2_l, EfficientNet_V2_L_Weights.DEFAULT, None),
]

models_default_weights_custom_imagesize_partial = [
    # ResNet models
    ("resnet18", resnet18, ResNet18_Weights.DEFAULT, (224, 224)),
    ("resnet34", resnet34, ResNet34_Weights.DEFAULT, (224, 224)),
    ("resnet50", resnet50, ResNet50_Weights.DEFAULT, (224, 224)),
    ("resnet101", resnet101, ResNet101_Weights.DEFAULT, (224, 224)),
    ("resnet152", resnet152, ResNet152_Weights.DEFAULT, (224, 224)),

    # EfficientNet B models
    ("efficientnet_b0", efficientnet_b0, EfficientNet_B0_Weights.DEFAULT, (224, 224)),
    ("efficientnet_b1", efficientnet_b1, EfficientNet_B1_Weights.DEFAULT, (240, 240)),
    ("efficientnet_b2", efficientnet_b2, EfficientNet_B2_Weights.DEFAULT, (288, 288)),
    ("efficientnet_b3", efficientnet_b3, EfficientNet_B3_Weights.DEFAULT, (300, 300)),

    # EfficientNet V2 models
    ("efficientnet_v2_s", efficientnet_v2_s, EfficientNet_V2_S_Weights.DEFAULT, None),
]

models_default_weights_partial = [
    # ResNet models
    ("resnet18", resnet18, ResNet18_Weights.DEFAULT, None),
    ("resnet34", resnet34, ResNet34_Weights.DEFAULT, None),
    ("resnet50", resnet50, ResNet50_Weights.DEFAULT, None),
    ("resnet101", resnet101, ResNet101_Weights.DEFAULT, None),
    ("resnet152", resnet152, ResNet152_Weights.DEFAULT, None),

    # EfficientNet B models
    ("efficientnet_b0", efficientnet_b0, EfficientNet_B0_Weights.DEFAULT, None),
    ("efficientnet_b1", efficientnet_b1, EfficientNet_B1_Weights.DEFAULT, None),
    ("efficientnet_b2", efficientnet_b2, EfficientNet_B2_Weights.DEFAULT, None),
    ("efficientnet_b3", efficientnet_b3, EfficientNet_B3_Weights.DEFAULT, None),

    # EfficientNet V2 models
    ("efficientnet_v2_s", efficientnet_v2_s, EfficientNet_V2_S_Weights.DEFAULT, None),
]

In [8]:
# 0 for single model training
# 1 for full model training without evaluation
# 2 for full model training with evaluation using default weights
# 3 for partial model training with evaluation using default weights
# 4 for full model training with evaluation using default weights and custom image sizes
# 5 for partial model training with evaluation using default weights and custom image sizes

MODEL_OPTION = 2
BATCH_SIZE = 32
NUM_EPOCHS = None

bulk_trainer = TrainTestBarkModel(sample_image_df, extract_path=extract_path)
bulk_trainer_and_eval = TrainTestBarkModel(tree_df=df_min_species_twelve, extract_path=extract_path, data_fraction=0.2)

if MODEL_OPTION == 0: # test if the training loop works with a single model before trying to train multiple models
    bulk_trainer.train_model(model_tuple=("resnet18", resnet18, (224, 224)), batch_size=BATCH_SIZE, num_epochs=NUM_EPOCHS)
elif MODEL_OPTION == 1: # test which models can train without running out of memory
    bulk_trainer.train_multiple(models=models_default_weights_full, batch_size=BATCH_SIZE, num_epochs=NUM_EPOCHS)
elif MODEL_OPTION == 2: # train and eval on all models using default weights
    bulk_trainer_and_eval.train_multiple(models=models_default_weights_full, batch_size=BATCH_SIZE, num_epochs=NUM_EPOCHS)
elif MODEL_OPTION == 3: # train and eval on partial model list with default weights
    bulk_trainer_and_eval.train_multiple(models=models_default_weights_partial, batch_size=BATCH_SIZE, num_epochs=NUM_EPOCHS)
elif MODEL_OPTION == 4: # train and eval on all models using default weights and custom image sizes
    bulk_trainer_and_eval.train_multiple(models=models_default_weights_custom_imagesize_full, batch_size=BATCH_SIZE, num_epochs=NUM_EPOCHS)
elif MODEL_OPTION == 5: # train and eval on partial model list with default weights and custom image sizes
    bulk_trainer_and_eval.train_multiple(models=models_default_weights_custom_imagesize_partial, batch_size=BATCH_SIZE, num_epochs=NUM_EPOCHS)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 139MB/s]



===== Training resnet18 =====
Using datafraction: 0.2, train_size: 0.7, val_size: 0.15, test_size: 0.15
Model loaded on device: cuda
Number of classes: 5
Epoch 1/6
Loss: 0.9080
Accuracy: 0.6650

Epoch 2/6
Loss: 0.4765
Accuracy: 0.8650

Epoch 3/6
Loss: 0.3549
Accuracy: 0.8750

Epoch 4/6
Loss: 0.1528
Accuracy: 0.9600

Epoch 5/6
Loss: 0.0552
Accuracy: 0.9800

Epoch 6/6
Loss: 0.0868
Accuracy: 0.9750

Validation Loss: 0.3329
Validation Accuracy: 0.9091

Test Loss: 1.6717
Test Accuracy: 0.5909

Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 149MB/s] 



===== Training resnet34 =====
Using datafraction: 0.2, train_size: 0.7, val_size: 0.15, test_size: 0.15
Model loaded on device: cuda
Number of classes: 5
Epoch 1/6
Loss: 0.9343
Accuracy: 0.6500

Epoch 2/6
Loss: 0.8147
Accuracy: 0.8200

Epoch 3/6
Loss: 0.5215
Accuracy: 0.7950

Epoch 4/6
Loss: 0.4749
Accuracy: 0.8600

Epoch 5/6
Loss: 0.2594
Accuracy: 0.8950

Epoch 6/6
Loss: 0.0585
Accuracy: 0.9900

Validation Loss: 0.8039
Validation Accuracy: 0.7955

Test Loss: 0.9420
Test Accuracy: 0.7500

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


...

100%|██████████| 208M/208M [00:01<00:00, 191MB/s] 



===== Training efficientnet_v2_m =====
Using datafraction: 0.2, train_size: 0.7, val_size: 0.15, test_size: 0.15
Model loaded on device: cuda
Number of classes: 5
Smoke test failed for efficientnet_v2_m: CUDA out of memory. Tried to allocate 118.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 5.81 MiB is free. Including non-PyTorch memory, this process has 14.55 GiB memory in use. Of the allocated memory 14.23 GiB is allocated by PyTorch, and 186.03 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)
Downloading: "https://download.pytorch.org/models/efficientnet_v2_l-59c71312.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_v2_l-59c71312.pth


100%|██████████| 455M/455M [00:03<00:00, 139MB/s] 



===== Training efficientnet_v2_l =====
Using datafraction: 0.2, train_size: 0.7, val_size: 0.15, test_size: 0.15
Model loaded on device: cuda
Number of classes: 5
Smoke test failed for efficientnet_v2_l: CUDA out of memory. Tried to allocate 450.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 403.81 MiB is free. Including non-PyTorch memory, this process has 14.17 GiB memory in use. Of the allocated memory 13.31 GiB is allocated by PyTorch, and 737.01 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)


,model,status,error
0,resnet18,passed,None
1,resnet34,passed,None
2,resnet50,passed,None
3,resnet101,passed,None
4,resnet152,passed,None
5,efficientnet_b0,passed,None
6,efficientnet_b1,passed,None
7,efficientnet_b2,passed,None
8,efficientnet_b3,passed,None
9,efficientnet_b4,failed,CUDA out of memory. Tried to allocate 48.00 Mi...
